In [ ]:
# Import necessary libraries for data analysis and visualization
import pandas as pd  # For data manipulation and analysis
import matplotlib.pyplot as plt  # For creating static visualizations
import seaborn as sns  # For advanced statistical visualizations built on top of Matplotlib
import numpy as np  #For working with arrays
import os  # For interacting with the operating system

In [2]:
#load data
demographic_df = pd.read_csv(r'C:\Users\Rithika\Downloads\Patient_demographics.csv')
diagnosis_df = pd.read_csv(r'C:\Users\Rithika\Downloads\Patient_diagnosis data.csv')

# Check the first few rows to ensure data is loaded correctly
print("Demographic Data:")
print(demographic_df.head())

print("\nDiagnostic Data:")
print(diagnosis_df.head())


Demographic Data:
   patient_id  age  gender marital_status education_level  occupation  \
0           1   70    male        widowed         college  unemployed   
1           2   34   other        married        graduate     student   
2           3   68   other        married     high school     retired   
3           4   47  female         single    postgraduate     student   
4           5   69  female        widowed     high school     student   

          income     smoking_status alcohol_consumption physical_activity  \
0  102511.985165         non-smoker               never              high   
1   30140.181211  occasional smoker              rarely               low   
2   28055.790819     regular smoker        occasionally               low   
3   57633.756818         non-smoker          frequently               low   
4   36972.500920         non-smoker          frequently         sedentary   

         bmi  family_history_of_cv  
0  23.784385                 False  
1  22.

In [3]:
# Merge the two DataFrames on patient_id
merged_df = pd.merge(demographic_df, diagnosis_df, on='patient_id')

# Check the merged DataFrame
print("\nMerged Data:")
print(merged_df.head())


Merged Data:
   patient_id  age  gender marital_status education_level  occupation  \
0           1   70    male        widowed         college  unemployed   
1           2   34   other        married        graduate     student   
2           3   68   other        married     high school     retired   
3           4   47  female         single    postgraduate     student   
4           5   69  female        widowed     high school     student   

          income     smoking_status alcohol_consumption physical_activity  \
0  102511.985165         non-smoker               never              high   
1   30140.181211  occasional smoker              rarely               low   
2   28055.790819     regular smoker        occasionally               low   
3   57633.756818         non-smoker          frequently               low   
4   36972.500920         non-smoker          frequently         sedentary   

   ...  diagnosis_date  systolic_bp diastolic_bp  cholesterol_total  \
0  ...       

In [4]:
# Drop the 'diagnosis_date' column from the merged DataFrame
merged_df.drop(columns=['diagnosis_date'],inplace=True)

#Display after deleting 
print("Updated DataFrame:")
print(merged_df.head())


Updated DataFrame:
   patient_id  age  gender marital_status education_level  occupation  \
0           1   70    male        widowed         college  unemployed   
1           2   34   other        married        graduate     student   
2           3   68   other        married     high school     retired   
3           4   47  female         single    postgraduate     student   
4           5   69  female        widowed     high school     student   

          income     smoking_status alcohol_consumption physical_activity  \
0  102511.985165         non-smoker               never              high   
1   30140.181211  occasional smoker              rarely               low   
2   28055.790819     regular smoker        occasionally               low   
3   57633.756818         non-smoker          frequently               low   
4   36972.500920         non-smoker          frequently         sedentary   

   ...  family_history_of_cv  systolic_bp  diastolic_bp  cholesterol_total  \
0

# Model Development

## Encoding 

In [5]:
import pandas as pd

# Apply one-hot encoding to 'smoking_status' and any other categorical columns if needed
merged_df = pd.get_dummies(merged_df, columns=['smoking_status'], drop_first=True)

# Display the updated DataFrame to check encoding
print(merged_df.head())

   patient_id  age  gender marital_status education_level  occupation  \
0           1   70    male        widowed         college  unemployed   
1           2   34   other        married        graduate     student   
2           3   68   other        married     high school     retired   
3           4   47  female         single    postgraduate     student   
4           5   69  female        widowed     high school     student   

          income alcohol_consumption physical_activity        bmi  ...  \
0  102511.985165               never              high  23.784385  ...   
1   30140.181211              rarely               low  22.833869  ...   
2   28055.790819        occasionally               low  30.612741  ...   
3   57633.756818          frequently               low  23.558659  ...   
4   36972.500920          frequently         sedentary  26.010163  ...   

   diastolic_bp  cholesterol_total  cholesterol_hdl  cholesterol_ldl  \
0            79         199.953980        50

## Scaling

In [6]:
from sklearn.preprocessing import MinMaxScaler

# Define the features you want to scale
features_to_scale = [
    'systolic_bp', 'diastolic_bp', 'cholesterol_total', 
    'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
    'blood_sugar', 'heart_rate', 'bmi'
]

# Include the one-hot encoded columns for 'smoking_status'
smoking_status_columns = [col for col in merged_df.columns if col.startswith('smoking_status_')]

# Combine all feature columns for scaling
all_features = features_to_scale + smoking_status_columns

# Initialize the MinMaxScaler to scale values between 0 and 1
# min max formula -->x-min(x)/max(x)-min(x)
min_max_scaler = MinMaxScaler()

# Fit and transform only the selected features, updating them in the DataFrame
merged_df[all_features] = min_max_scaler.fit_transform(merged_df[all_features])

# Print the scaled features
print(merged_df[all_features])


       systolic_bp  diastolic_bp  cholesterol_total  cholesterol_hdl  \
0         0.089888      0.487179           0.483826         0.523389   
1         0.134831      0.282051           0.448496         0.374774   
2         0.404494      0.794872           0.503929         0.699945   
3         0.910112      0.794872           0.268140         0.491500   
4         0.202247      0.358974           0.571977         0.338742   
...            ...           ...                ...              ...   
19995     0.674157      0.564103           0.557019         0.622904   
19996     0.865169      0.230769           0.443298         0.442667   
19997     0.438202      0.051282           0.486876         0.449990   
19998     0.089888      0.230769           0.688410         0.607930   
19999     0.887640      0.897436           0.383628         0.704064   

       cholesterol_ldl  triglycerides  blood_sugar  heart_rate       bmi  \
0             0.457972       0.506500     0.501436    0.615

## Risk Categorization

In [7]:
def categorize_risk_scaled(row):
    if (row['systolic_bp'] > 1.0 or row['diastolic_bp'] > 1.0 or  # Adjust these thresholds as per your scaling
        row['cholesterol_total'] > 1.0 or 
        row['cholesterol_hdl'] < -1.0 or  # Note: for scaled values, lower numbers indicate lower levels
        row['cholesterol_ldl'] > 1.0 or 
        row['triglycerides'] > 1.0 or 
        row['blood_sugar'] > 1.0 or 
        row['bmi'] > 1.0 or 
        row['smoking_status_regular smoker'] == 1 or 
        row['smoking_status_occasional smoker'] == 1):
        return 'High Risk'
    else:
        return 'Low Risk'
# Apply the function to create the 'risk_category' column
merged_df['risk_category'] = merged_df.apply(categorize_risk_scaled, axis=1)

# Check the balance of the target variable
print(merged_df['risk_category'].value_counts())

# Encode the risk category into binary values
merged_df['risk_category_encoded'] = merged_df['risk_category'].map({'High Risk': 1, 'Low Risk': 0})

# Display the updated DataFrame with the new columns
print(merged_df[['patient_id', 'risk_category', 'risk_category_encoded']])


High Risk    13356
Low Risk      6644
Name: risk_category, dtype: int64
       patient_id risk_category  risk_category_encoded
0               1      Low Risk                      0
1               2     High Risk                      1
2               3     High Risk                      1
3               4      Low Risk                      0
4               5      Low Risk                      0
...           ...           ...                    ...
19995       19996     High Risk                      1
19996       19997     High Risk                      1
19997       19998     High Risk                      1
19998       19999     High Risk                      1
19999       20000      Low Risk                      0

[20000 rows x 3 columns]


## Handling Class Imbalance Using SMOTE

In [38]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import RFE

# Check class distribution
y=merged_df['risk_category']
class_distribution = y.value_counts()
print("Class distribution in target variable:")
print(class_distribution)

# Define the features you want to include for training
features_to_include = ['systolic_bp', 'diastolic_bp', 'cholesterol_total', 
                       'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
                       'blood_sugar', 'heart_rate', 'bmi']

# Prepare the features DataFrame
X = merged_df[features_to_include]

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training + validation and testing sets (80% train + validation, 20% test)
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Further split the temporary set into training and validation sets (75% train, 25% validation of the temp set)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Check the sizes of the datasets and class distributions
print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print("Class distribution in training set:")
print(y_train.value_counts())
print("Class distribution in validation set:")
print(y_val.value_counts())

# Apply SMOTE only if both classes are present in the training set
if len(y_train.value_counts()) > 1:
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

    # Check the sizes of the original and balanced training sets
    print("Original training set size:", X_train.shape[0])
    print("Balanced training set size:", X_train_balanced.shape[0])
else:
    print("Not enough classes to apply SMOTE. Proceeding without balancing.")


Class distribution in target variable:
High Risk    13356
Low Risk      6644
Name: risk_category, dtype: int64
Training set size: 12000
Validation set size: 4000
Test set size: 4000
Class distribution in training set:
High Risk    8014
Low Risk     3986
Name: risk_category, dtype: int64
Class distribution in validation set:
High Risk    2671
Low Risk     1329
Name: risk_category, dtype: int64
Original training set size: 12000
Balanced training set size: 16028


## Logistic Regression

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Define feature columns
features_to_include = ['systolic_bp', 'diastolic_bp', 'cholesterol_total', 
                       'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
                       'blood_sugar', 'heart_rate', 'bmi']

# Prepare features and target
X = merged_df[features_to_include]
y = merged_df['risk_category']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Initialize Logistic Regression model
logistic_model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
logistic_model.fit(X_train, y_train)

# Make predictions on the test set
y_test_pred = logistic_model.predict(X_test)

# Evaluate the model
print("Confusion Matrix (2x2):")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))



Confusion Matrix (2x2):
[[1318 1353]
 [ 669  660]]

Classification Report:
              precision    recall  f1-score   support

   High Risk       0.66      0.49      0.57      2671
    Low Risk       0.33      0.50      0.39      1329

    accuracy                           0.49      4000
   macro avg       0.50      0.50      0.48      4000
weighted avg       0.55      0.49      0.51      4000



## Decision Tree

In [10]:
from sklearn.tree import DecisionTreeClassifier

features_to_include = ['systolic_bp', 'diastolic_bp', 'cholesterol_total', 
                       'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
                       'blood_sugar', 'heart_rate', 'bmi']

# Prepare the features DataFrame
X = merged_df[features_to_include]

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Fit a Decision Tree Classifier
decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train, y_train)

# Make predictions on the test set
y_test_pred = decision_tree_model.predict(X_test)

# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))


Confusion Matrix:
[[1715  956]
 [ 839  490]]

Classification Report:
              precision    recall  f1-score   support

   High Risk       0.67      0.64      0.66      2671
    Low Risk       0.34      0.37      0.35      1329

    accuracy                           0.55      4000
   macro avg       0.51      0.51      0.50      4000
weighted avg       0.56      0.55      0.56      4000



## Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

features_to_include = ['systolic_bp', 'diastolic_bp', 'cholesterol_total', 
                       'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
                       'blood_sugar', 'heart_rate', 'bmi']

# Prepare the features DataFrame
X = merged_df[features_to_include]

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Fit a Random Forest Classifier
random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train, y_train)

# Make predictions on the test set
y_test_pred = random_forest_model.predict(X_test)

# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))


Confusion Matrix:
[[2623   48]
 [1301   28]]

Classification Report:
              precision    recall  f1-score   support

   High Risk       0.67      0.98      0.80      2671
    Low Risk       0.37      0.02      0.04      1329

    accuracy                           0.66      4000
   macro avg       0.52      0.50      0.42      4000
weighted avg       0.57      0.66      0.54      4000



## Gradient Boosting Classifier

In [49]:
from sklearn.ensemble import GradientBoostingClassifier

# Define the features and target variable
features_to_include = ['systolic_bp', 'diastolic_bp', 'cholesterol_total', 
                       'cholesterol_hdl', 'cholesterol_ldl', 'triglycerides', 
                       'blood_sugar', 'heart_rate', 'bmi']
X = merged_df[features_to_include]
y = merged_df['risk_category']  # Target variable

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ensure that both X_train and X_test are Pandas DataFrames
X_train = pd.DataFrame(X_train, columns=features_to_include)
X_test = pd.DataFrame(X_test, columns=features_to_include)

# Initialize and fit the Gradient Boosting model
gradient_boosting = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
gradient_boosting.fit(X_train, y_train)

# Make predictions on the test set
y_test_pred = gradient_boosting.predict(X_test)

# Evaluate the model
print("Confusion Matrix for Gradient Boosting:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report for Gradient Boosting:")
print(classification_report(y_test, y_test_pred, zero_division=0))


Confusion Matrix for Gradient Boosting:
[[2671    0]
[314   1015]]

Classification Report for Gradient Boosting:
              precision    recall  f1-score   support

   High Risk       0.68      0.99      0.80      2721
    Low Risk       0.29      0.01      0.02      1279

    accuracy                           0.67      4000
   macro avg       0.49      0.50      0.41      4000
weighted avg       0.56      0.67      0.56      4000



## K-Nearest Neighbors (KNN)

In [13]:
from sklearn.neighbors import KNeighborsClassifier

# Fit a K-Nearest Neighbors model
knn_model = KNeighborsClassifier()
knn_model.fit(X_train, y_train)

# Make predictions on the test set
y_test_pred_knn = knn_model.predict(X_test)

# Evaluate the model
print("Confusion Matrix for KNN:")
print(confusion_matrix(y_test, y_test_pred_knn))
print("\nClassification Report for KNN:")
print(classification_report(y_test, y_test_pred_knn, zero_division=0))


Confusion Matrix for KNN:
[[2099  572]
 [1065  264]]

Classification Report for KNN:
              precision    recall  f1-score   support

   High Risk       0.66      0.79      0.72      2671
    Low Risk       0.32      0.20      0.24      1329

    accuracy                           0.59      4000
   macro avg       0.49      0.49      0.48      4000
weighted avg       0.55      0.59      0.56      4000



## Gradient Boosting, a supervised learning algorithm is selected as it performs well compared to other models with an accuracy of 67% and a more favorable confusion matrix, demonstrating superior classification of both High Risk and Low Risk categories.